In [1]:
# if the following command generates an error, you probably didn't enable 
# the cluster security option "Allow API access to all Google Cloud services"
# under Manage Security → Project Access when setting up the cluster
!gcloud dataproc clusters list --region us-central1

NAME          PLATFORM  PRIMARY_WORKER_COUNT  SECONDARY_WORKER_COUNT  STATUS   ZONE           SCHEDULED_DELETE  SCHEDULED_STOP
cluster-0016  GCE       2                                             RUNNING  us-central1-a


# Imports & Setup

In [ ]:
# !pip install -q google-cloud-storage==1.43.0
!pip install google-cloud-storage>=2.10.0
!pip install -q graphframes

In [ ]:
import pyspark
import sys
import pickle
import hashlib
from google.cloud import storage

import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
# if nothing prints here you forgot to include the initialization script when starting the cluster
!ls -l /usr/lib/spark/jars/graph*

-rw-r--r-- 1 root root 247882 Dec  7 17:24 /usr/lib/spark/jars/graphframes-0.8.2-spark3.1-s_2.12.jar


In [10]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark import SparkContext, SparkConf, SparkFiles
from pyspark.sql import SQLContext
from graphframes import *

In [11]:
spark

In [ ]:
# Put your bucket name below and make sure you can access it without an error
bucket_name = '322793365' 
full_path = f"gs://{bucket_name}/"
paths=[]

client = storage.Client()
blobs = client.list_blobs(bucket_name)
for b in blobs:
    if b.name != 'graphframes.sh':
        paths.append(full_path+b.name)

***GCP setup is complete!*** 

# Building an inverted index

Here, we read the entire corpus to an rdd, directly from Google Storage Bucket and use your code from Colab to construct an inverted index.

In [ ]:
parquetFile = spark.read.parquet(f"gs://{bucket_name}/multistream*")

doc_text_pairs  = parquetFile.select("text", "id").rdd
doc_title_pairs = parquetFile.select("title", "id").rdd
doc_anchor_text_pairs = parquetFile.select("anchor_text", "id").rdd

flattened_doc_anchor_pairs = doc_anchor_text_pairs.flatMap(
    lambda rec: [(a.text, a.id) for a in rec[0] if getattr(a, "text", None)]
)

We will count the number of pages to make sure we are looking at the entire corpus. The number of pages should be more than 6M

In [17]:
# Count number of wiki pages
parquetFile.count()

6348910

Let's import the inverted index module. Note that you need to use the staff-provided version called `inverted_index_gcp.py`, which contains helper functions to writing and reading the posting files similar to the Colab version, but with writing done to a Google Cloud Storage bucket.

In [ ]:
# adding our python module to the cluster
sc.addFile("/home/dataproc/inverted_index_gcp.py")
sc.addFile("/home/dataproc/indexer.py")
sys.path.insert(0,SparkFiles.getRootDirectory())

25/12/07 17:37:53 WARN org.apache.spark.SparkContext: The path /home/dataproc/inverted_index_gcp.py has been added already. Overwriting of added paths is not supported in the current version.


In [ ]:
from inverted_index_gcp import InvertedIndex
from indexer import (
    word_count,
    reduce_word_counts,
    calculate_df,
    partition_postings_and_write,
    construct_inverted_index,
    write_inverted_index,
    generate_graph,
    process_page_views,
    create_title_mappings,
    create_embeddings,
    RE_WORD,
    all_stopwords
)

In [ ]:
text_index = construct_inverted_index(
    doc_text_pairs,
    client,
    bucket_name, 
    index_prefix='postings_gcp/text',
    apply_filter=True
)
index_src, index_dst = write_inverted_index(text_index, client, bucket_name, 'text_index', index_prefix='postings_gcp/text')

In [ ]:
title_index = construct_inverted_index(
    doc_title_pairs, 
    client,
    bucket_name,
    index_prefix='postings_gcp/title',
    apply_filter=False
)
index_src, index_dst = write_inverted_index(title_index, client, bucket_name,'title_index', index_prefix='postings_gcp/title')

In [ ]:
# Build anchor index
anchor_index = construct_inverted_index(
    flattened_doc_anchor_pairs,
    client,
    bucket_name,
    index_prefix='postings_gcp/anchor',
    apply_filter=False
)
index_src, index_dst = write_inverted_index(anchor_index, client, bucket_name, 'anchor_index', index_prefix='postings_gcp/anchor')

# PageRank

In [ ]:
print("Computing PageRank...")
pages_links = spark.read.parquet(f"gs://{bucket_name}/multistream*").select("id", "anchor_text").rdd

# Construct the graph 
edges, vertices = generate_graph(pages_links)

# Compute PageRank
edgesDF = edges.toDF(['src', 'dst']).repartition(124, 'src')
verticesDF = vertices.toDF(['id']).repartition(124, 'id')
g = GraphFrame(verticesDF, edgesDF)
pr_results = g.pageRank(resetProbability=0.15, maxIter=6)
pr = pr_results.vertices.select("id", "pagerank")
pr = pr.sort(col('pagerank').desc())
pr.repartition(1).write.csv(f'gs://{bucket_name}/pr', compression="gzip")
pr.show()
print("✅ PageRank computed and uploaded")

+-------+------------------+
|     id|          pagerank|
+-------+------------------+
|3434750| 9913.728782160773|
|  10568| 5385.349263642038|
|  32927| 5282.081575765277|
|  30680| 5128.233709604119|
|5843419| 4957.567686263868|
|  68253|  4769.27826535516|
|  31717|  4486.35018054831|
|  11867|4146.4146509127695|
|  14533|3996.4664408855037|
| 645042|3531.6270898037424|
|  17867|3246.0983906041415|
|5042916| 2991.945739166177|
|4689264| 2982.324883041747|
|  14532| 2934.746829203171|
|  25391| 2903.546223513398|
|   5405| 2891.416329154636|
|4764461| 2834.366987332661|
|  15573| 2783.865118158839|
|   9316|2782.0396464137693|
|8569916| 2775.286191840016|
+-------+------------------+
only showing top 20 rows



# Page views

In [ ]:
print("Processing page views...")
pv_path = 'https://dumps.wikimedia.org/other/pageview_complete/monthly/2021/2021-08/pageviews-202108-user.bz2'

# Process and upload page views
page_view_dict = process_page_views(pv_path, bucket_name=bucket_name)

print(f"✅ Processed {len(page_view_dict)} documents with page view data")

# Title Mappings

In [ ]:
# Create and upload title mappings
title_dict = create_title_mappings(doc_title_pairs, bucket_name=bucket_name)

# Show a few examples
print("\nExample mappings:")
for doc_id, title in list(title_dict.items())[:5]:
    print(f"  {doc_id}: {title}")

# embedding indecis

In [ ]:
!pip install gensim

In [ ]:
doc_ids, embeddings = create_embeddings(doc_title_pairs, bucket_name, RE_WORD, all_stopwords)